[![Buy Me A Coffee](https://img.shields.io/badge/Buy%20Me%20A%20Coffee-support%20my%20work-FFDD00?style=flat&labelColor=101010&logo=buy-me-a-coffee&logoColor=white)](https://www.buymeacoffee.com/r0mymendez)

---

# 📑 Docling: structured document extraction

One of the main challenges when working with scientific `PDFs` is that they are not “simple” documents. They are full of **tables**, **columns**, **formulas**, **figures**, and complex `layouts` that are not always preserved correctly when text is extracted.

**IBM Docling** is an open source library designed for `PDF` extraction and document structuring. Its goal is not only to extract text, but also to convert complex documents into a **structured representation** that can be used in artificial intelligence pipelines and `RAG` systems.
Instead of returning messy plain text, **Docling** tries to preserve the structure of the document, including the **reading order**, **tables**, **formulas**, **images**, and other key elements of the content.
The following image summarizes some of the key benefits of using Docling for complex document processing.

![05-docling-key-features](img/05-docling-key-features.png)

---

## Why use Docling?

Traditional tools like `PyPDF`, `PDFPlumber`, or classic `OCR` are usually enough for simple documents, but they often fail when working with **scientific papers** or documents with complex `layouts`.
In these cases, important information can be lost, such as:
* 〰️ **table structure**
* 〰️ **column separation**
* 〰️ **relationship between text and figures**
* 〰️ **mathematical formulas**

**Docling** appears as an alternative that tries to solve exactly these problems, generating a much more consistent output for later analysis.

---

## Docling features
Below, you can find the main features published by the library on **Hugging Face**:
* 🏷️ DocTags for Efficient Tokenization – Introduces DocTags an efficient and minimal representation * for documents that is fully compatible with DoclingDocuments.
* 🔍 OCR (Optical Character Recognition) – Extracts text accurately from images.
* 📐 Layout and Localization – Preserves document structure and document element bounding boxes.
* 💻 Code Recognition – Detects and formats code blocks including identation.
* 🔢 Formula Recognition – Identifies and processes mathematical expressions.
* 📊 Chart Recognition – Extracts and interprets chart data.
* 📑 Table Recognition – Supports column and row headers for structured table extraction.
* 🖼️ Figure Classification – Differentiates figures and graphical elements.
* 📝 Caption Correspondence – Links captions to relevant images and figures.
* 📜 List Grouping – Organizes and structures list elements correctly.
* 📄 Full-Page Conversion – Processes entire pages for comprehensive document conversion including all page elements (code, equations, tables, charts etc.)
* 🔲 OCR with Bounding Boxes – OCR regions using a bounding box.
* 📂 General Document Processing – Trained for both scientific and non-scientific documents.


# 🏥 Practical example: processing a medical record with Docling

In this notebook we will work with a **synthetically generated clinical history** in PDF format.
All patient data, medical records, and clinical findings are entirely fictional and created 
for educational purposes only. No real patient information was used.

This document represents a real-world use case in the healthcare industry, where 
medical records need to be processed, structured, and made available for AI-powered analysis.

## What we will cover

* 〰️ **Step 1** — Load and convert the PDF using Docling
* 〰️ **Step 2** — Explore the document structure and identify sections
* 〰️ **Step 3** — Extract structured patient data into a pandas DataFrame

---

> 📌 Below you can see part of the clinical history we will be working with.  
> In the following exercise we will identify and extract specific sections of this document.

<br>

![](img/EHR.png)

# 📑 Loading and Converting the PDF

In this step we load the clinical history PDF using Docling's `DocumentConverter`. 
Docling automatically detects the document structure and exports the result in two formats:

* 〰️ `Markdown`: human-readable output for previewing content
* 〰️ `Dictionary`: programmatic access to text, tables, images and metadata

This structured output is what makes Docling more powerful than a basic PDF text extractor.

In [ ]:
from docling.document_converter import DocumentConverter, PdfFormatOption
import pandas as pd
converter = DocumentConverter()


result = converter.convert("clinical_history_structured.pdf")
# export markdown
data_markdown = result.document.export_to_markdown()

# export dict
data_dict = result.document.export_to_dict()
texts = data_dict['texts']

2026-05-25 21:01:02,666 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]


2026-05-25 21:01:02,672 - INFO - Going to convert document batch...
2026-05-25 21:01:02,672 - INFO - Initializing pipeline for StandardPdfPipeline with options hash e15bc6f248154cc62f8db15ef18a8ab7
2026-05-25 21:01:02,673 - INFO - Auto OCR model selected ocrmac.
2026-05-25 21:01:02,674 - INFO - Accelerator device: 'mps'
2026-05-25 21:01:03,725 - INFO - Accelerator device: 'mps'
2026-05-25 21:01:04,564 - INFO - Processing document clinical_history_structured.pdf
2026-05-25 21:01:06,878 - INFO - Finished converting document clinical_history_structured.pdf in 4.22 sec.


# 🗂️ Exploring Document Sections

Every clinical document is organized into sections. Here we extract all **section headers** 
detected by Docling — such as `Patient Identification`, `Chief Complaint`, and `Laboratory Results`.

This gives us:

* 〰️ A **map of the document structure**
* 〰️ The ability to **target specific sections** for downstream processing

In [57]:
[item['text'] for item in data_dict['texts'] 
                 if item['label'] == 'section_header']

['CITYVIEW MEDICAL CENTER CLINICAL HISTORY AND RECORD',
 '1. PATIENT IDENTIFICATION',
 '4. PAST MEDICAL HISTORY',
 '5. MEDICATIONS',
 '6. ALLERGIES',
 '2. CHIEF COMPLAINT',
 '3. HISTORY OF PRESENT ILLNESS',
 '7. FAMILY HISTORY',
 '8. SOCIAL HISTORY',
 '9. REVIEW OF SYSTEMS',
 '10. PHYSICAL EXAMINATION',
 '12. LABORATORY RESULTS',
 '13. ASSESSMENT',
 '14. PLAN',
 '11. IMAGING']

# 🧩 Extracting Patient Data as a Structured Table

Now we extract the content of the first section — **Patient Identification** — by filtering 
items that belong to `#/groups/0`. Docling preserves the key-value layout of the original PDF, 
so we can split the flat list into field names and values using Python slice notation.

The result is a clean `pandas DataFrame` ready for:

* 〰️ Analysis
* 〰️ Storage
* 〰️ Downstream AI processing

In [ ]:
# Filter group 0
group_0 = [item['orig'] for item in texts 
           if item.get('parent', {}).get('$ref') == '#/groups/0']

# Convertir lista plana a pares clave/valor
keys   = group_0[0::2]  # índices pares
values = group_0[1::2]  # índices impares

df = pd.DataFrame({
    'field': [k.replace(':', '').strip() for k in keys],
    'value': values
})

df

,field,value
0,Full Name,John Michael Doe
1,Date of Birth,"April 23, 1978 (46 y/o)"
2,Gender,Male
3,Address,"1254 Oak Street, Apt 5B Springfield, IL 62701"
4,Phone,(217) 555-0198
5,Email,johndoe@email.com
6,Marital Status,Married
7,Occupation,Software Engineer
8,Primary Language,English
9,Insurance,HealthPlus PPO


In [9]:
result.document.tables[0].export_to_dataframe()

2026-05-25 21:01:54,343 - WARNING - Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.


,Medication,Dose,Route,Frequency,Indication
0,Lisinopril,10 mg,PO,Once daily,Hypertension
1,Atorvastatin,20 mg,PO,Once daily (at night),Dyslipidemia
2,Acetaminophen,500 mg,PO,PRN,Fever / Pain
